# الدرس 07 - نمط تصميم التخطيط

توضح هذه الدفترية نمط تصميم **التخطيط** لوكلاء الذكاء الاصطناعي باستخدام إطار عمل Microsoft Agent.
ستتعلم كيفية تقسيم طلبات السفر المعقدة إلى مهام فرعية منظمة، وتعيينها إلى وكلاء متخصصين،
وتنفيذ الخطة الناتجة — كل ذلك باستخدام المخرجات المنظمة المدعومة بنماذج Pydantic.


## الإعداد


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## تفكيك المهمة

يعتبر تفكيك المهمة جوهر نمط تصميم التخطيط. بدلاً من مطالبة وكيل واحد بالتعامل مع طلب معقد من البداية إلى النهاية، نقوم بتقسيم المشكلة إلى **مهام فرعية** أصغر ومحددة جيداً. يتم تعيين كل مهمة فرعية إلى وكيل متخصص (مثل الرحلات، الفنادق، الأنشطة) مع تحديد أولويات واضحة وترتيب التبعية.

يوفر هذا النهج عدة فوائد:
- **الوضوح**: كل مهمة فرعية لها مسؤولية واحدة فقط.
- **التوازي**: يمكن للمهمات الفرعية المستقلة أن تعمل بالتزامن.
- **الموثوقية**: تكون الأخطاء معزولة في المهام الفرعية الفردية.
- **تتبع الميزانية**: تُقدّر التكاليف لكل مهمة فرعية وتُجمع فيما بعد.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## إنشاء وكيل تخطيط مع إخراج منظم

يعمل وكيل التخطيط كـ **منسق استقبال**. عند استلام طلب سفر على مستوى عالٍ، ينتج وكيل التخطيط خطة سفر منظمة `TravelPlan` — يقوم بتفكيك الطلب إلى مهام فرعية، تحديد الأولويات، وتحديد التبعيات بحيث يمكن للكُونسيرج أو طبقة التنفيذ إنجاز العمل.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## تنفيذ خطة باستخدام أدوات متخصصة

بمجرد أن يقوم موظف الاستقبال بإعداد خطة منظمة، يقوم **وكيل الكونسيرج** بتنفيذها.
كل أداة متخصصة تتعامل مع فئة واحدة من المهام الفرعية (الرحلات، الفنادق، الأنشطة). يقوم الكونسيرج بتكرار المهام الفرعية في الخطة حسب ترتيب الاعتماد، ويرسل كل منها إلى الأداة المناسبة.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## الملخص

في هذا الدرس تعلمت **نمط تصميم التخطيط** لوكلاء الذكاء الاصطناعي:

1. **تفكيك المهمة** — وكيل التخطيط في الاستقبال يقوم بتقسيم طلب السفر المعقد إلى مهام فرعية منظمة باستخدام نماذج Pydantic، مع تعيين كل مهمة لوكيل متخصص مع تحديد الأولويات والاعتمادات.
2. **الإخراج المنظم** — من خلال تمرير `response_format` يقوم الوكيل بإرجاع كائن `TravelPlan` تم التحقق من صحته بدلاً من النص الحر، مما يجعل المعالجة اللاحقة موثوقة.
3. **تنفيذ الخطة** — يقوم وكيل الكونسيرج بالتكرار عبر المهام الفرعية باستخدام الأدوات المتخصصة (`book_flight`، `reserve_hotel`، `book_activity`) لتنفيذ الخطة وتقديم النتائج.

يفصل هذا النمط بين *ماذا نفعل* (التخطيط) و*كيف نفعل ذلك* (التنفيذ)، مما يجعل الوكلاء أكثر مرونة وقابلين للاختبار وأسهل في التوسيع.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:  
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى جاهدين لتحقيق الدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار النسخة الأصلية من المستند بلغتها الأصلية المصدر الموثوق والمعتمد. بالنسبة للمعلومات الحساسة أو الهامة، يُنصح بالاعتماد على الترجمة البشرية الاحترافية. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
